# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR² Kilifi Mental Health dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and contains survey data from Kilifi County, Kenya, focusing on mental health indicators. This notebook demonstrates metadata extraction, record set exploration, extraction via Croissant IDs, and basic data analysis and visualization.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n")
print(f"Dataset Description: {metadata.description}\n")

## 2. Data Overview
Enumerate available record sets, fields, and their `@id` values. This informs you of the different data tables and variables present for further extraction.

We use the `dataset.record_sets` attribute to get all available record sets, then for each, list the fields and columns, *always* referencing each by its `@id`.

In [ ]:
# List all record sets by @id and their schema
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"\nRecord Set @id: {rs.id}\n  name: {rs.name}")
    print("  Fields and columns (@id):")
    for field in rs.fields:
        field_id = field.id
        print(f"    - field @id: {field_id}  |  label: {getattr(field, 'label', field.name) if hasattr(field, 'label') else field.name}")
        if hasattr(field, 'columns'):
            for col in field.columns:
                print(f"         - column @id: {col.id}  |  label: {getattr(col, 'label', col.name) if hasattr(col, 'label') else col.name}")

## 3. Data Extraction
Load data for one or more record sets into pandas DataFrames for analysis. Always specify record sets and fields by their `@id`.

Replace the example record set and field IDs below with the ones listed above for your analysis.

In [ ]:
# For demonstration, extract all available record sets by their @id
dataframes = {}
for rs in dataset.record_sets:
    rs_id = rs.id
    # Load records for this record set
    records = list(dataset.records(record_set=rs_id))
    try:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(dataframes[rs_id])} rows in DataFrame for record set {rs_id}")
        print(f"Fields (@id): {list(dataframes[rs_id].columns)}\n")
    except Exception as e:
        print(f"Could not load DataFrame for record set {rs_id}: {e}\n")
# Pick one main record set for further demonstration below (replace with yours if needed)
main_record_set = next(iter(dataframes.keys()))  # Use the first as example
print(f"Using main record_set: {main_record_set}")
dataframes[main_record_set].head(3)

## 4. Exploratory Data Analysis (EDA)
Perform operations such as filtering records, normalizing numeric fields, and grouping by attributes.

Below, replace `numeric_field_id` and `group_field_id` with actual field `@id`s observed above. If the chosen field does not exist, replace with a field from your DataFrame.

Example fields might include composite scores such as `GAD-7_total` or `PHQ-9_total`.

In [ ]:
# Example: Analyze a numeric field (replace with an actual @id from exploration above)
df = dataframes[main_record_set]

# Guess candidate columns (user may replace these if schema changes):
candidate_numeric_fields = [c for c in df.columns if (
    (c.lower().endswith('_total') or ('score' in c.lower()) or (df[c].dtype.kind in 'fi'))
)]
print('Candidate numeric fields:', candidate_numeric_fields)
if candidate_numeric_fields:
    numeric_field_id = candidate_numeric_fields[0]  # take first match
else:
    numeric_field_id = df.columns[0]  # fallback

# Filter for records above an arbitrary threshold
threshold = 10
if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")
    print(filtered_df[[numeric_field_id]].head(3))

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))
else:
    print(f"{numeric_field_id} is not numeric; EDA skipped.")

# Group by a categorical field (replace with an @id if you know it, e.g. 'gender', 'education', etc.)
candidate_group_fields = [c for c in df.columns if df[c].dtype == object and not c.startswith('@')]
print(f"Group candidates: {candidate_group_fields}")
if candidate_group_fields and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    group_field = candidate_group_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
    print(f"\nGrouped mean {numeric_field_id} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the chosen numeric field (boxplot/histogram) and the mean per group (if applicable).

Adjust the field or group names by their DataFrame column (`@id`) if needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=16)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot grouped by a field
    if candidate_group_fields:
        plt.figure(figsize=(9, 4))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
- This notebook demonstrated loading metadata and records from the FAIR² Croissant dataset using entity `@id`s for all references.
- We extracted main survey data and demonstrated exploratory analysis of numeric scores (e.g., GAD-7, PHQ-9, or other summary fields), normalized values, and visualized distributions.
- These techniques are adaptable to any Croissant-compliant dataset—simply enumerate the available record sets and use their `@id` for downstream data science workflows.